1. 首先我们需要进行环境配置,我们需要安装uv来设置我们的开发环境,我是用的是Ubuntu 24.04,安装uv使用的是这个命令:`curl -LsSf https://astral.sh/uv/install.sh | sh`,uv 安装结束之后创建我们的工作目录,我的工作目录名称是`learn-langChain-langGraph`,创建好工作目录后,我们进入到这个工作目录,执行命令`uv init -p 3.13`来初始化我们的工作环境.

2. 初始化好工作环境后,我们添加三个库`uv add notebook` `uv add openai` `uv add python-dotenv`然后开始我们的学习

3. 执行下面的代码来测试,需要注意的一点是我们需要在项目的根目录中创建一个文件,名称是`.env`文件内容是我们的apikey,内容示例:`API_KEY=sk-fxxxxxxxx362exxxx`,然后执行下面的代码

## 单次对话代码示例

In [3]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("API_KEY")
#print(api_key)

#创建openai客户端

client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com"
)
response = client.chat.completions.create(
    model="deepseek-v4-pro",
    messages=[
        {"role": "system", "content": "你是一名渗透测试专家，专注于网络安全和漏洞分析。我是你的上司,我会给你一些任务，你需要根据我的指示进行渗透测试，并提供详细的分析和建议。"},
        {"role": "user", "content": "什么是Linux本地提权"},
    ],
    stream=False,
    reasoning_effort="high",
    extra_body={"thinking": {"type": "enabled"}}
)
print("Response:",response)
print(response.choices[0].message.content)


Response: ChatCompletion(id='87fb09c2-db31-4382-b2a2-a06852ece9f9', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Linux 本地提权（Local Privilege Escalation）是指攻击者在已获得目标系统低权限用户访问（如普通用户或服务账户）后，利用系统或应用程序的漏洞、错误配置，将该权限提升为 root 或高权限用户的过程。这是渗透测试攻击链中的关键环节，通常发生在初始立足之后，目的是获取系统的完全控制权。\n\n作为渗透测试人员，你需要掌握 Linux 本地提权的攻击面、技术手法和防御检测思路。以下是详细的专业分析：\n\n### 1. Linux 本地提权的常见攻击向量\n\n#### 1.1 内核漏洞提权\n- **原理**：利用 Linux 内核代码中的内存损坏漏洞（如 Use-After-Free、Buffer Overflow）获取内核态代码执行，从而直接修改进程凭证（cred 结构体）提权。\n- **经典案例**：Dirty COW（CVE-2016-5195）、Dirty Pipe（CVE-2022-0847）、OverlayFS 漏洞（CVE-2021-3493 等）。\n- **手法**：\n  - 查找内核版本：`uname -a; cat /etc/os-release`\n  - 公开利用代码搜索（Exploit-DB、GitHub），编译并执行。\n- **检测/规避**：补丁管理是关键。现代 EDR 会监控内核异常调用和利用行为。渗透测试时，应优先尝试非内核提权方法，避免触发告警或导致系统崩溃。\n\n#### 1.2 服务/应用程序漏洞与错误配置\n- **原理**：以 root 运行的服务（如 Web 服务器、数据库、备份脚本）存在命令注入、反序列化或任意文件写入漏洞，从而导致代码以 root 权限执行。\n- **典型场景**：\n  - MySQL UDF 提权：需要具备 MySQL root 密码和 `secure_file_priv` 配置可写。\n  - 计划任务（cron）通配符注入：

## 多轮对话代码示例

In [2]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("API_KEY")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com"
)

# 第一轮
messages = [
    {"role": "system", "content": "你是一名渗透测试专家，专注于网络安全和漏洞分析。我是你的上司,我会给你一些任务，你需要根据我的指示进行渗透测试，并提供详细的分析和建议。"},
    {"role": "user", "content": "什么是Linux本地提权"}
]
response = client.chat.completions.create(
    model="deepseek-v4-pro",
    messages=messages
)

# ✅ 正确：转字典 + 正确访问 response
messages.append({
    "role": "assistant",
    "content": response.choices[0].message.content
})
print(f"Round 1 回复: {response.choices[0].message.content}")

# 第二轮
messages.append({"role": "user", "content": "早期的提权方式有哪些?"})
response = client.chat.completions.create(
    model="deepseek-v4-pro",
    messages=messages
)

messages.append({
    "role": "assistant",
    "content": response.choices[0].message.content
})
print(f"Round 2 回复: {response.choices[0].message.content}")

Round 1 回复: Linux 本地提权（Local Privilege Escalation）是指攻击者在已经获得目标 Linux 系统低权限用户（如普通用户或服务账户）Shell 后，利用系统配置缺陷、内核漏洞、应用程序漏洞或不当权限设置，将自身权限提升至更高等级（通常是 root）的过程。这是渗透测试中关键的一环，能帮助攻击者从受限访问转变为完全控制系统。

---

## 常见 Linux 本地提权方法

### 1. 内核漏洞提权
利用 Linux 内核中已知的漏洞，通过执行精心构造的利用代码（Exploit）直接获取 root 权限。例如有名的：
- Dirty COW (CVE-2016-5195)
- Dirty Pipe (CVE-2022-0847)
- OverlayFS 相关漏洞 (CVE-2021-3493 等)

检测可利用的内核漏洞通常使用工具如：
- Linux Exploit Suggester
- linuxprivchecker.py
- unix-privesc-check

### 2. SUID/SGID 提权
具有 SUID 位的可执行文件在执行时会以文件所有者的权限运行。若 root 所拥有的二进制文件设置了 SUID，且该文件存在命令注入、环境变量利用或可写性等问题，就能被提权利用。

典型利用场景：
- 可通过 `GTFOBins` 查询常见 SUID 二进制利用方法（如 find, vim, less, more, bash, python 等）。
- 若 SUID 程序调用了外部命令而未使用完整路径，可通过 PATH 劫持注入恶意程序。

查找 SUID 文件：  
```bash
find / -perm -4000 -type f 2>/dev/null
```

### 3. Sudo 配置错误
`/etc/sudoers` 中若配置不当，普通用户可以以 root 身份执行特定命令，而这些命令可能被滥用获取 root shell。

典型示例：
- `sudo` 允许执行 vim，然后通过 `:!bash` 跳出到 root shell。
- sudo 允许无密码执行某程序，或允许执行任意编辑器、脚本等。

查看当前用户 sudo 权限：
```bash
sudo -l
```

### 4. Cro